# Baseline Training

In [ ]:
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import netCDF4
from pathlib import Path
from datetime import datetime
import json
import torch

## Config

In [22]:
IN_DIR = Path("/projects/dagr5998/icepack_array_output/test2_output") 
SPLIT_DIR = Path("/projects/dagr5998/icepack_split_output/test2_split")

In [24]:
def make_index(run_ids, T, input_window):
    # build the indicies for the input series, have step incase we want over
    index = []
    for r in run_ids:
        t0 = 0
        while t0 + input_window <= T:
            index.append([int(r), t0])
            t0 += input_window
    return index

class IcepackDataset(torch.utils.data.Dataset):
    def __init__(self, X, Y, index, window, mu, sd):
        self.X = X# (N, T, C)
        self.Y = Y# (N, P)
        self.index = index #array of [run, start]
        self.window = window
        self.mu, self.sd = mu,sd  

    def __getitem__(self, ind):
        run,start = self.index[ind]
        xnorm = self.X[run, start:start + self.window]# (Window, C)
        return (xnorm - self.mu)/self.sd, self.Y[run]

    def __len__(self):
        return len(self.index)

#### Loading Data

In [17]:
mu,sd = np.load(IN_DIR / 'mu.npy'), np.load (IN_DIR/'sd.npy')
mu,sd = torch.from_numpy(mu), torch.from_numpy(sd)

In [30]:
with open(IN_DIR / 'config.json') as f:
    config = json.load(f)
with open(SPLIT_DIR / 'split_config.json') as f:
    split_config = json.load(f)
var_ind_list = split_config["var_ind_list"]

In [26]:
var_ind_list

[10, 13, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39]

In [32]:
X = np.load(IN_DIR / "X.npy") # input tensor
Y = np.load(IN_DIR / "Y.npy") # output tensor
train_index = (IN_DIR / "train_index.npy")
test_index  = (IN_DIR / "test_index.npy")

In [33]:
Xt = torch.tensor(X[:,:,var_ind_list])
Yt = torch.tensor(Y)

In [34]:
Xt.shape
Yt.shape

torch.Size([140, 1])

#### Non-ML Checks